In [1]:
import sys
import pandas as pd
import pyexcel as pe
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import matplotlib.pyplot as plt


# ANSI escape code for output formatting
BOLD = "\033[1m" if "idlelib" not in sys.modules else ""
RESET = "\033[0m" if "idlelib" not in sys.modules else ""

# set global decimal display format
pd.set_option('display.float_format', '{:.4f}'.format)

"""
Load and clean the dataset, assign normal (1) and abnormal (0) class labels to the NSP column
"""

# load dataset
data = pe.get_sheet(file_name='CTG.xls', sheet_name='Raw Data').to_array()
df = pd.DataFrame(data[1:], columns=data[0])
df = df[['LB', 'ALTV', 'Min', 'Mean', 'NSP']]  # selected feature set

# convert empty cells to Na, then drop rows with empty cells
df = df.replace(r'^\s*$', pd.NA, regex=True).convert_dtypes()
df = df.dropna()

# assign normal (1) and abnormal (0) classes
df['NSP'] = df['NSP'].apply(lambda x: 1 if x == 1 else 0)

print(f'{BOLD}Overview of the Dataset (Selected Features){RESET}\n',df)

print(f'\n{BOLD}Dataset Summary{RESET}\n',df.describe())

corr = df.corr()
print(f'\n{BOLD}Examination of Feature Correlations{RESET}\n', corr)

"""
Split the data 50/50, which will be used for all the models
Build a Naive Bayesian classifier to predict the class label, compute accuracy and confusion matrix
"""
# split dataset
X = df[['LB', 'ALTV', 'Min', 'Mean']]
y = df['NSP']

seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=seed)


def build_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    # confusion matrix
    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn) if (tp + fn) else 0
    tnr = tn / (tn + fp) if (tn + fp) else 0

    result = {'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
              'accuracy': acc, 'TPR': tpr, 'TNR': tnr}

    return acc, pd.DataFrame([result], index=[model_name])


nb_acc, nb_result = build_model('naive bayesian', GaussianNB(), X_train, X_test, y_train, y_test)

print(f'{BOLD}Naive Bayesian Model{RESET}\n'
      f'Accuracy: {nb_acc:.4f} \n\n'
      f'Confusion Matrix Metrics:\n',
      nb_result)


"""
Build a decision tree classifier to predict the class label, compute accuracy and confusion matrix
"""
dt_acc, dt_result = build_model('decision tree', DecisionTreeClassifier(random_state=seed),
                                X_train, X_test, y_train, y_test)

print(f'{BOLD}Decision Tree Model{RESET}\n'
      f'Accuracy: {dt_acc:.4f} \n\n'
      f'Confusion Matrix Metrics:\n',
      dt_result)


"""
 Build random forest classifiers with number of estimators ranging from 1 to 10, and maximum depth ranging
            from 1 to 5, find the optimal model, and compute its accuracy and confusion matrix
"""

error_matrix = np.zeros((10, 5))  # matrix to store error rate

for n in range(1, 11):
    for d in range(1, 6):
        model = RandomForestClassifier(
            n_estimators=n,
            max_depth=d,
            criterion='entropy',
            random_state=seed
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        error = 1 - accuracy_score(y_test, y_pred)
        error_matrix[n - 1, d - 1] = error


# plot error rate and combination of N & d
plt.figure(figsize=(10, 6))
for i in range(5):
    plt.plot(range(1, 11), error_matrix[:, i], label=f"max_depth={i+1}")

plt.xlabel('n_estimator (N)')
plt.ylabel('Error Rate')
plt.title('Random Forest Error Rates')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print('\n\nBy observing the error rate graph, it is evident that the model performance improves as'
      '\nboth the number of trees (N) and max depth (d) increases.'
      '\nThe combination with the lowest error rate is N = 10, d=5\n')


# calculate accuracy and confusion for the optimal combination
rf_model = RandomForestClassifier(
    n_estimators=10,
    max_depth=5,
    criterion='entropy',
    random_state=seed
)

rf_acc, rf_result = build_model('random forest', rf_model, X_train, X_test, y_train, y_test)

print(f'{BOLD}Random Forest Model with Optimal N & d{RESET}\n'
      f'Accuracy: {rf_acc:.4f} \n\n'
      f'Confusion Matrix Metrics:\n',
      rf_result)


"""
Compare performance metrics of the 3 models, and discuss findings
"""
stats_df = pd.concat([nb_result, dt_result, rf_result])
print(f'{BOLD}Model Performance{RESET}\n',
        stats_df)


print(f'\n{BOLD}Observations:{RESET}'
      '\nThe random forest model has the highest accuracy of 0.8899, followed by decision tree at 0.8514;'
      '\nnaive bayes has the lowest accuracy at 0.8319.  This is consistent with how each model handles complexity'
      '\nand feature interactions.  Random forest, through its bagging ensemble method, was able to reduce'
      '\noverfitting and capture more subtle patterns, leading to more accurate predictions compared to a single'
      '\ndecision tree.  Naive Bayse, on the other hand, assumes strong feature independence, and therefore lead to'
      '\nmore misclassifications when the data exhibits complex or correlated patterns, as seen in this dataset.'

      '\n\nIn terms of TPR, random forest ranks the highest at 0.953, followed by naive bayes at 0.9131, and'
      '\ndecision tree at 0.8391.  This indicates that all models are generally good at identifying the normal class,'
      '\nwhere random forest performs best due to its ensemble nature, reducing variance and improving generalization.'
      '\nAlthough naive bayse ranked the second place for TPR, it is observed that the model tend to assign more normal'
      '\nclass label. Since the dataset is highly imbalanced, with normal class accounting for over 75% of the samples,'
      '\nthe probabilistic nature of naive bayse causes the model to favour the majority class, resulting a higher TPR,'
      '\nhowever,at the expense of accuracy and TNR.'

      '\n\nFor the TNR, decision tree performs best at 0.6838, followed by random forest at 0.6667, and naive bayes at'
      '\n0.5769.  Both decision tree and random forest employ adaptive feature-based splitting, allowing them to better'
      '\ncapture feature interactions and class boundaries.  Decision tree slightly outperforms random forest in TNR,'
      "\nwhich can be attributed to the model's stricter and more extreme splits, resulting in better isolation of the"
      "\nabnormal class, but at the cost of overfitting and generalization.  Meanwhile, the bagging mechanism of random"
      "\nforest trains multiple trees  trains multiple trees on random data subsets, and its ensemble averaging softens"
      "\ndecision boundaries, resulting in a more generalized but slightly less sharp separation between classes.")


FileNotFoundError: [Errno 2] No such file or directory: 'CTG.xls'